# LogSAD — Live Inference Demo
**Category: `juice_bottle` | Protocol: 4-shot**

Runs on the **SKKU GPU server `aicoss220`** (RTX 4090).

| Item | Location on server |
|---|---|
| LogSAD repo | `/home/gaya6/LogSAD/` |
| `demo/images/` (8 images) | `demo/images/` (already in repo) |
| `sam_vit_h_4b8939.pth` (2.4 GB) | `checkpoint/sam_vit_h_4b8939.pth` |
| CLIP + DINOv2 weights | auto-downloaded by HuggingFace on first run |
| conda env | `logsad` (`/home/gaya6/miniconda3/envs/logsad`) |

> **Demo environment (Group 6).**
> Open this notebook on the GPU server via **VS Code Remote-SSH** (`aicoss220`) — connect from the classroom machine (31709), select the `logsad` kernel, then run.
>
> **Pre-execute before presenting:** Sections 0–3 (imports → model load → 4-shot memory-bank build, ~3–5 min — keep the kernel warm).
> Sections 4–7 are the ~1-minute live portion (marked **▶ LIVE FROM HERE**).
> The saved cell outputs below also act as a fallback if the kernel disconnects.

## 0. Server Setup

**Kernel:** select `logsad` in VS Code (path: `/home/gaya6/miniconda3/envs/logsad/bin/python`).

This cell sets the working directory to the repo root so all local modules (`model_ensemble_few_shot`, `open_clip_local`, etc.) are importable.

In [ ]:
#OWN CODE — set working directory to repo root so local modules are importable
import os

# Notebook lives in demo/; move up one level to the repo root.
# If the kernel is already at the repo root (e.g. re-run), this is a no-op.
if os.path.basename(os.getcwd()) == 'demo':
    os.chdir('..')

print('Working directory:', os.getcwd())
print('GPU server: aicoss220 | conda env: logsad')

## 1. Imports

In [ ]:
#OWN CODE — demo configuration (category / k-shot / image paths)
import sys
sys.path.insert(0, os.getcwd())

import torch
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms

from model_ensemble_few_shot import MyModel

CATEGORY  = 'juice_bottle'
K_SHOT    = 4
IMAGE_DIR = 'demo/images'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {device}')
print(f'Category: {CATEGORY}  |  Protocol: {K_SHOT}-shot')
# model_ensemble_few_shot.py calls matplotlib.use('Agg') on import, which would
# suppress inline figures in the notebook. Re-enable the inline backend so plots render.
from IPython import get_ipython
get_ipython().run_line_magic('matplotlib', 'inline')


## 2. 4-Shot Normal Reference Images
These 4 images define what "normal" looks like — they replace a training dataset.

In [ ]:
#OWN CODE — load & visualise the 4 normal reference images (= the entire "training" set)
to_tensor = transforms.Compose([
    transforms.Resize((448, 448)),
    transforms.ToTensor(),
])

train_paths = [f'{IMAGE_DIR}/train/good/{i:03d}.png' for i in range(K_SHOT)]
train_imgs  = [Image.open(p).convert('RGB') for p in train_paths]

fig, axes = plt.subplots(1, K_SHOT, figsize=(4 * K_SHOT, 4))
fig.suptitle(f'{CATEGORY} — {K_SHOT} normal reference images (= entire "training" data)', fontsize=13)
for i, (img, ax) in enumerate(zip(train_imgs, axes)):
    ax.imshow(img)
    ax.set_title(f'Normal #{i+1}')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Model Initialization (4-shot)
Loads CLIP ViT-L/14, DINOv2 ViT-L/14, SAM ViT-H — all **frozen**.
Builds the memory bank from the 4 reference images. No gradient computation.

> Expected time: ~3–5 min (dominated by model loading)

In [ ]:
model = MyModel()
model.to(device)
model.eval()

model.setup({
    'few_shot_samples':      torch.stack([to_tensor(img) for img in train_imgs]).to(device),
    'few_shot_samples_path': train_paths,
    'dataset_category':      CATEGORY,
})
print('Memory bank built. Model ready.')

---
## ▶ LIVE FROM HERE — run these cells during the presentation
Everything **above** this line is executed **before** the talk (model loading + 4-shot memory bank, ~3–5 min, kernel kept warm — this is the training-free analogue of "training", and exceeds the 2–3 min live budget, so it is done in advance).

The cells **below** are the ~1-minute live portion: **image selection → inference → anomaly-map visualisation.** Their saved outputs serve as a fallback if anything disconnects.

## 4. Select Test Images
4 representative images:
- 1 **normal**
- 2 **logical anomalies** (liquid color does not match the fruit label tag)
- 1 **structural anomaly** (physical defect on the bottle)

In [ ]:
#OWN CODE — pick representative test images (normal / logical / structural) for the live demo
selected = [
    (f'{IMAGE_DIR}/test/good/000.png',                 'Normal',             0),
    (f'{IMAGE_DIR}/test/logical_anomalies/000.png',    'Logical Anomaly #1', 1),
    (f'{IMAGE_DIR}/test/logical_anomalies/001.png',    'Logical Anomaly #2', 1),
    (f'{IMAGE_DIR}/test/structural_anomalies/000.png', 'Structural Anomaly', 1),
]

type_colors = {'Normal': 'green', 'Logical Anomaly #1': 'red',
               'Logical Anomaly #2': 'red', 'Structural Anomaly': 'orange'}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Test images selected for inference', fontsize=13)
for ax, (path, label, _) in zip(axes, selected):
    ax.imshow(np.array(Image.open(path).resize((448, 448))) / 255.0)
    ax.set_title(label, color=type_colors[label], fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Run Inference
Each image passes through three detectors:
- **Patch Matching** (CLIP + DINOv2 PatchCore) → structural score
- **Interest Matching** (SAM + Hungarian algorithm) → interest-level score
- **Composition Matching** (CLIP zero-shot) → logical score

In [ ]:
#OWN CODE — demo inference harness wrapping the paper's model.forward()
results = []
for path, label_str, gt_label in selected:
    img_pil    = Image.open(path).convert('RGB')
    img_tensor = to_tensor(img_pil).unsqueeze(0)

    with torch.no_grad():
        output = model(img_tensor.to(device), [path])

    pred_score  = output['pred_score'].item()
    anomaly_map = output['anomaly_map'].cpu()
    decision    = 'ANOMALY' if pred_score > 0.5 else 'NORMAL'
    correct     = '\u2713' if (decision == 'ANOMALY') == (gt_label == 1) else '\u2717'

    results.append((img_pil, label_str, gt_label, pred_score, anomaly_map, decision, correct))
    print(f'[{label_str:<22}]  score={pred_score:.4f}  ->  {decision}  {correct}')

## 6. Anomaly Map Visualization
- Warm colours (red/yellow) = high anomaly score
- Cool colours (blue) = normal region

In [ ]:
#OWN CODE — anomaly-map visualisation (input / heat-map / overlay)
def visualize(img_pil, anomaly_map, pred_score, gt_label_str, decision, correct):
    img_np = np.array(img_pil.resize((448, 448))) / 255.0
    h, w   = img_np.shape[:2]

    amap = F.interpolate(
        anomaly_map.unsqueeze(0).unsqueeze(0).float(),
        size=(h, w), mode='bilinear', align_corners=True
    ).squeeze().numpy()
    amap_norm = (amap - amap.min()) / (amap.max() - amap.min() + 1e-8)

    title_color = 'green' if decision == 'NORMAL' else 'red'

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
    axes[0].imshow(img_np)
    axes[0].set_title(f'Input  (GT: {gt_label_str})', fontsize=11)
    axes[0].axis('off')

    im = axes[1].imshow(amap_norm, cmap='jet', vmin=0, vmax=1)
    axes[1].set_title('Anomaly Map', fontsize=11)
    axes[1].axis('off')
    plt.colorbar(im, ax=axes[1], fraction=0.046)

    axes[2].imshow(img_np)
    axes[2].imshow(amap_norm, cmap='jet', alpha=0.45, vmin=0, vmax=1)
    axes[2].set_title(
        f'Overlay  |  Score: {pred_score:.4f}\nPrediction: {decision}  {correct}',
        fontsize=11, color=title_color, fontweight='bold'
    )
    axes[2].axis('off')
    plt.tight_layout()
    plt.show()


for img_pil, label_str, gt_label, pred_score, anomaly_map, decision, correct in results:
    visualize(img_pil, anomaly_map, pred_score, label_str, decision, correct)

## 7. Score Summary

In [ ]:
#OWN CODE — per-image score summary + paper-vs-ours AUROC line
import pandas as pd

summary = pd.DataFrame([
    {
        'Image Type': label_str,
        'GT Label':   'Anomalous' if gt == 1 else 'Normal',
        'Pred Score': f'{score:.4f}',
        'Decision':   decision,
        'Correct':    correct,
    }
    for _, label_str, gt, score, _, decision, correct in results
])

print(summary.to_string(index=False))
print()
print('-' * 55)
print('Paper 4-shot AUROC (juice_bottle) : 84.3')
print('Ours  4-shot AUROC (juice_bottle) : 84.26')